# Import

In [ ]:
import os
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format

TRAIN_PATH = '/kaggle/input/politeness/politeness_train.csv'
VAL_PATH = '/kaggle/input/politeness/politeness_val.csv'
TEST_PATH = '/kaggle/input/politeness/politeness_test.csv'

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

# Package Version Info

In [ ]:
import sys, platform
from importlib.metadata import version, PackageNotFoundError

pkgs = [
    "optuna",
    "scikit-learn",
    "transformers",
    "sentence-transformers",
]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

for p in pkgs:
    try:
        print(f"{p}: {version(p)}")
    except PackageNotFoundError:
        print(f"{p}: NOT INSTALLED")

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
import re

LINK_PATTERN = re.compile(r"http\S+|www\.\S+")
TAG_PATTERN  = re.compile(r"#\w+")
MENT_PATTERN = re.compile(r"@\w+")
TATWEEL_CHAR = "ـ"
STRONG_PUNC  = re.compile(r"[!؟\?]{2,}")
ELONGATION   = re.compile(r"(.)\1\1+")
DIAC_PATTERN = re.compile(r'[\u064B-\u0652]')

ADDRESSEE_MARKERS = [
    "يا ",
    "يـا ",
    "ياخي",
    "يا اخي",
    "يا أخي",
    "يااختي",
    "يا اختي",
    "يا أختي",
    "حضرتك",
    "سيدي",
    "أستاذ",
    "استاذ",
    "دكتور",
    "د.",
    "مهندس",
    "معالي",
    "سعادة",
    "انت",
    "انتي",
    "أنتم",
    "حضراتكم",
    "يا جماعة",
    "يا سادة",
    "ياسادة"
]

POLITE_PHRASES = [
    "شكرا",
    "شكرًا",
    "شكراً",
    "يعطيك",
    "يعطيكم",
    "لو سمحت",
    "من فضلك",
    "فضلاً",
    "فضلا",
    "الله يبارك",
    "بارك الله",
    "الله يجزاك خير",
    "ربي يوفقك",
    "ربي يحفظك",

    "الله ي",

    "تسلم",
    "مشكور",
    "جزيل الشكر",
    "تكرم",
    "لو تكرمت",
    "لو تكرمتي",
    "ما قصرت",
    "ماقصرت",
    "بيض الله وجهك",
    "جزاك",
    "جزاكم",
    "بارك الله",
    "الله يعطيكم العافية",
    "مشكورين",
    "مشكورين على",
]

RESPECT_TITLES = [
    "استاذ",
    "أستاذ",
    "دكتور",
    "د.",
    "شيخ",
    "سيدي",
    "حضرتك",
    "أخي",
    "أختي",
    "بروف",
    "بروفيسور",
    "مهندس",
    "م.",
    "سعادة",
    "معالي"
]

INSULT_WORDS = [
    "غباء",
    "كلب",
    "حقير",
    "وسخ",
    "مخيس",
    "زفت",
    "سرقة",
    "قرف",
    "كذاب",
    "ساقط",
    "سفلة",
    "قرف",
    "فاشل",
    "فاشلة",
    "الهبد",
    "خايس",
    "زباله ",
    "زبالة ",
    "لعن ",
    "عفن",
    "مصخرة",
    "انجس",
    "زي الزفت",
    "زي الح",
    "زي وجهكم",
    "زي وجهك",
    "سفل",
    "خائن",
    "دلاخه",
    "كلاب",
    "👎",
    "🤬",
    "خسي",
    "لعنة ",
    "مستفز",
    "عبيط",
    "طز",
    "يا غبي",
    "قذر",
    "فشله",
    "الله لا يوفقكك",
    "حسبي الله"
]

import json
with open("/kaggle/input/auto-lexicons/automated_lexicons_.json", 'r', encoding='utf-8') as f:
    loaded_data = json.load(f)

INSULT_WORDS_AUTO = loaded_data["IMPOLITE_WORDS"]
NEUTRAL_WORDS_AUTO = loaded_data["NEUTRAL_WORDS"]
POLITE_PHRASES_AUTO = loaded_data["POLITE_WORDS"]

# Exploratory Data Analysis (EDA)

In [ ]:
# Hidden to avoid cluttering the notebook :)
def dataframe_information(df, dataframe_name=''):
    '''
        Basic Dataframe informations we need to know:
        Shape, Columns, Data types, Missing values
    '''
    # Getting the relavent information
    shape = df.shape
    columns = df.columns.tolist()

    # Printing them :)
    print()
    print('-'*40)
    print(f'Dataframe Information: {dataframe_name}')
    print('-'*40)
    print(f'Dataset Shape: {shape}\nColumn Names: {columns}')

    # Print a sample of the dataframe
    print('Sample:')
    display(df.head(1))
    print()

In [ ]:
dataframe_information(train_df, 'Train')
dataframe_information(valid_df, 'Valid')
dataframe_information(test_df, 'Test')

In [ ]:
import re

def clean_arabic_for_politeness(text):
    # Remove Diacritics
    tashkeel_pattern = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(tashkeel_pattern, '', text)

    # Normalize Alef, Hamza, and Ya
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Reduce Letter Repetition (Lengthening)
    text = re.sub(r'(.)\1+', r'\1', text)

    return text.strip()

# Apply this BEFORE you run the Log-Odds keyword extraction
train_df['Clean_Sentence'] = train_df['Sentence'].apply(clean_arabic_for_politeness)
valid_df['Clean_Sentence'] = valid_df['Sentence'].apply(clean_arabic_for_politeness)
test_df['Clean_Sentence'] = test_df['Sentence'].apply(clean_arabic_for_politeness)

In [ ]:
import emoji
import numpy as np

def get_emojis(sentence):
    return [e["emoji"] for e in emoji.emoji_list(str(sentence))]

def count_phrases(text, phrases):
    t = str(text)
    return sum(1 for p in phrases if p in t)

def count_emojis(s, emo_set):
    s = str(s)
    return sum(1 for ch in s if ch in emo_set)

def starts_with_any(text, phrases):
    t = str(text).strip()
    return int(any(t.startswith(p) for p in phrases))

def sync_keywords_to_cleaning(keyword_list):
    cleaned_list = [clean_arabic_for_politeness(word) for word in keyword_list]
    return list(set(cleaned_list))

def detect_elongation(text):
    if re.search(r'(.)\1\1+', str(text)):
        return 1
    return 0

def extract_advanced_pragmatics(df):
    def get_features(text):
        text = str(text)

        emojis_found = emoji.emoji_count(text)
        repeated_punc = 1 if re.search(r'([!؟?])\1+', text) else 0
        matches = DIAC_PATTERN.findall(text)
        count = len(matches)

        return pd.Series({
            'emoji_count': emojis_found,
            'repeated_punc': repeated_punc,
            'exclamation_count': text.count('!'),
            'arabic_question_count': text.count('؟'),
            'has_diacritics': 1 if count > 0 else 0,
            'diacritic_count': count,
            'diacritic_density': count / len(text) if len(text) > 0 else 0
        })

    # Apply to dataframe
    new_feats = df['Sentence'].apply(get_features)
    return pd.concat([df, new_feats], axis=1)

def build_eda_features(df, dataframe_name=''):
    n_df = df.copy()
    s = n_df["Sentence"].astype(str)
    s_clean = n_df["Clean_Sentence"].astype(str)

    n_df["polite_phrase_cnt"] = s.map(lambda x: count_phrases(x, POLITE_PHRASES))
    n_df["insult_cnt"]        = s.map(lambda x: count_phrases(x, INSULT_WORDS))
    n_df["respect_title_cnt"] = s.map(lambda x: count_phrases(x, RESPECT_TITLES))
    n_df["addressee_cnt"]   = s.map(lambda x: count_phrases(x, ADDRESSEE_MARKERS))
    n_df["has_addressee"] = (n_df["addressee_cnt"] > 0).astype(int)
    n_df['has_elongation'] = n_df['Sentence'].apply(detect_elongation)

    n_df["polite_phrase_auto_cnt"] = s.map(lambda x: count_phrases(x, POLITE_PHRASES_AUTO))
    n_df["insult_auto_cnt"]        = s.map(lambda x: count_phrases(x, INSULT_WORDS_AUTO))
    n_df["neutral_auto_cnt"]        = s.map(lambda x: count_phrases(x, NEUTRAL_WORDS_AUTO))

    POLITE_PHRASES_CLEAN = sync_keywords_to_cleaning(POLITE_PHRASES)
    INSULT_WORDS_CLEAN = sync_keywords_to_cleaning(INSULT_WORDS)
    RESPECT_TITLES_CLEAN = sync_keywords_to_cleaning(RESPECT_TITLES)
    ADDRESSEE_MARKERS_CLEAN = sync_keywords_to_cleaning(ADDRESSEE_MARKERS)
    n_df["polite_phrase_cnt_clean"] = s_clean.map(lambda x: count_phrases(x, POLITE_PHRASES_CLEAN))
    n_df["insult_cnt_clean"]        = s_clean.map(lambda x: count_phrases(x, INSULT_WORDS_CLEAN))
    n_df["respect_title_cnt_clean"] = s_clean.map(lambda x: count_phrases(x, RESPECT_TITLES_CLEAN))
    n_df["addressee_cnt_clean"]   = s_clean.map(lambda x: count_phrases(x, ADDRESSEE_MARKERS_CLEAN))
    n_df["has_addressee_clean"] = (n_df["addressee_cnt_clean"] > 0).astype(int)

    INSULT_WORDS_AUTO_CLEAN = sync_keywords_to_cleaning(INSULT_WORDS_AUTO)
    POLITE_PHRASES_AUTO_CLEAN = sync_keywords_to_cleaning(POLITE_PHRASES_AUTO)
    NEUTRAL_WORDS_AUTO_CLEAN = sync_keywords_to_cleaning(NEUTRAL_WORDS_AUTO)
    n_df["polite_phrase_auto_cnt_clean"] = s_clean.map(lambda x: count_phrases(x, INSULT_WORDS_AUTO_CLEAN))
    n_df["insult_auto_cnt_clean"]        = s_clean.map(lambda x: count_phrases(x, POLITE_PHRASES_AUTO_CLEAN))
    n_df["neutral_auto_cnt_clean"]        = s_clean.map(lambda x: count_phrases(x, NEUTRAL_WORDS_AUTO_CLEAN))

    n_df = extract_advanced_pragmatics(n_df)
    return n_df

In [ ]:
train_n_df = build_eda_features(train_df, 'Train')
valid_n_df = build_eda_features(valid_df, 'Valid')
test_n_df = build_eda_features(test_df, 'Test')

# Baseline Models (LR)

In [ ]:
from sklearn.model_selection import train_test_split

X_train = train_df['Sentence']
y_train = train_df['label']

X_valid = valid_df["Sentence"]
y_valid = valid_df["label"]

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score, precision_recall_fscore_support, f1_score
LABEL_ORDER = ["Impolite", "Neutral", "Polite"]

def _safe_col(label):
    # make safe column names like "f1__Impolite"
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(label))

def compute_metrics_full(y_true, y_pred, labels):
    acc = accuracy_score(y_true, y_pred)

    # per-class
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )

    # macro + weighted
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    out = {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f_macro,
        "precision_weighted": p_w,
        "recall_weighted": r_w,
        "f1_weighted": f_w,
    }

    # add per-class fields
    for lab, pi, ri, fi, si in zip(labels, p, r, f1, sup):
        labc = _safe_col(lab)
        out[f"precision__{labc}"] = pi
        out[f"recall__{labc}"] = ri
        out[f"f1__{labc}"] = fi
        out[f"support__{labc}"] = int(si)

    return out

def embed(model_name, texts_train, texts_valid, batch_size=64):
    st = SentenceTransformer(model_name, device="cuda")
    Xtr = st.encode(
        texts_train.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
    )
    Xva = st.encode(
        texts_valid.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
    )
    return Xtr, Xva

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f_macro,
        "precision_weighted": p_w,
        "recall_weighted": r_w,
        "f1_weighted": f_w,
    }

def fit_eval_lr(Xtr, ytr, Xva, yva, labels, C=1.0, penalty="l2", solver="lbfgs"):
    kwargs = dict(
        C=C,
        penalty=penalty,
        solver=solver,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )
    if solver != "liblinear":
        kwargs["n_jobs"] = -1

    clf = LogisticRegression(**kwargs)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xva)

    metrics = compute_metrics_full(yva, pred, labels=labels)
    return metrics, clf, pred

def lr_grid_search(Xtr, ytr, Xva, yva, labels,
                   C_list=(0.25, 0.5, 1, 2, 4, 8),
                   solvers=("lbfgs", "liblinear")):
    rows = []
    best = (None, None, None)  # (metrics, params, pred)

    for solver in solvers:
        penalties = ("l2",) if solver == "lbfgs" else ("l1", "l2")

        for penalty in penalties:
            for C in C_list:
                metrics, _, pred = fit_eval_lr(Xtr, ytr, Xva, yva, labels=labels, C=C, penalty=penalty, solver=solver)
                params = {"C": C, "penalty": penalty, "solver": solver}
                rows.append({**params, **metrics})

                if (best[0] is None) or (metrics["f1_macro"] > best[0]["f1_macro"]):
                    best = (metrics, params, pred)

    grid_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
    return grid_df, best

In [ ]:
models = {
    "UBC-NLP/MARBERTv2": "UBC-NLP/MARBERTv2",
    "MarBERT-Triplet-Matryoshka": "Omartificial-Intelligence-Space/Marbert-all-nli-triplet-Matryoshka",
    "GATE-AraBert-v1": "Omartificial-Intelligence-Space/GATE-AraBert-v1",
    "Arabic-Triplet-Matryoshka-V2": "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
    "Arabic-all-nli-Triplet-Matryoshka": "Omartificial-Intelligence-Space/Arabic-all-nli-triplet-Matryoshka",
}
all_runs = []
best_rows = []

for name, model_id in models.items():
    print("-"*80)
    print(name, "->", model_id)

    Xtr_emb, Xva_emb = embed(model_id, X_train, X_valid, batch_size=64)

    grid_df, (best_metrics, best_params, best_pred) = lr_grid_search(
        Xtr_emb, y_train, Xva_emb, y_valid,
        labels=LABEL_ORDER,
        C_list=(0.25, 0.5, 1, 2, 4, 8),
        solvers=("lbfgs", "liblinear")
    )

    grid_df["model"] = name
    grid_df["model_id"] = model_id
    all_runs.append(grid_df)

    best_rows.append({
        "model": name,
        "model_id": model_id,
        **best_params,
        **best_metrics
    })

best_summary = pd.DataFrame(best_rows).sort_values("f1_macro", ascending=False)
all_runs_df = pd.concat(all_runs, ignore_index=True)
best_summary.to_csv("baseline_results_summary.csv", index=False)

In [ ]:
paper_cols = ["model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "C", "penalty", "solver"]
display(best_summary[paper_cols])

per_class_cols = ["model"]
for lab in LABEL_ORDER:
    c = _safe_col(lab)
    per_class_cols += [f"precision__{c}", f"recall__{c}", f"f1__{c}", f"support__{c}"]

# Feature Fusion (Embeddings + Handcrafted Features) + LR

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

MODEL_ID = "Omartificial-Intelligence-Space/Marbert-all-nli-triplet-Matryoshka"
st = SentenceTransformer(MODEL_ID, device="cuda")

clf = LogisticRegression(
    C=0.5, penalty="l1", solver="liblinear",
    max_iter=5000, class_weight="balanced", random_state=42
)

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

In [ ]:
Xtr_sent = st.encode(train_df["Sentence"].astype(str).tolist(),
                     batch_size=32, show_progress_bar=True)

Xva_sent = st.encode(valid_df["Sentence"].astype(str).tolist(),
                     batch_size=32, show_progress_bar=True)

ytr = train_df["label"].values
yva = valid_df["label"].values

In [ ]:
src_tr = pd.get_dummies(train_df["Source"], prefix="src")
src_va = pd.get_dummies(valid_df["Source"], prefix="src").reindex(columns=src_tr.columns, fill_value=0)
src_te = pd.get_dummies(test_df["Source"], prefix="src").reindex(columns=src_tr.columns, fill_value=0)

Xtr_src = csr_matrix(src_tr.values)
Xva_src = csr_matrix(src_va.values)
Xte_src = csr_matrix(src_te.values)

## Add Dialect + Intent + Sarcasm

In [ ]:
from transformers import pipeline
import pandas as pd
import torch

# Dialect Model
did_pipe = pipeline("text-classification",
                    model="IbrahimAmin/marbertv2-arabic-written-dialect-classifier",
                    device=0)

# Intent Model
intent_pipe = pipeline("text-classification",
                       model="bassemessam/Arabic-bank77-intent-classification",
                       device=0)
# Sarcasm Model
sar_pipe = pipeline(
    "text-classification",
    model="hardiksr/sarcasm-classifier-bert-base-arabic-camelbert-msa-data",
    device=0,
    torch_dtype=torch.float16
)

def add_sarcasm(df, col="Sentence", bs=32):
    texts = df[col].astype(str).tolist()
    out = sar_pipe(texts, truncation=True, padding=True, batch_size=bs)
    df["sarcasm_score"] = [r["score"] if "sarc" in r["label"].lower() else 1-r["score"] for r in out]
    return df


def extract_dialect_features(df):
    texts = df['Sentence'].astype(str).tolist()
    results = did_pipe(texts, truncation=True, padding=True, batch_size=32)
    df['dial_label'] = [res['label'] for res in results]
    df['dial_conf'] = [res['score'] for res in results]
    return df

def extract_intent_features(df):
    texts = df['Sentence'].astype(str).tolist()
    results = intent_pipe(texts, padding=True, truncation=True, batch_size=32)
    df['intent_label'] = [res['label'] for res in results]
    df['intent_conf'] = [res['score'] for res in results]
    return df

# Apply to Train
train_n_df = extract_dialect_features(train_n_df)
train_n_df = extract_intent_features(train_n_df)
train_n_df = add_sarcasm(train_n_df)

# Apply to Validation
valid_n_df = extract_dialect_features(valid_n_df)
valid_n_df = extract_intent_features(valid_n_df)
valid_n_df = add_sarcasm(valid_n_df)


# Apply to Test
test_n_df = extract_dialect_features(test_n_df)
test_n_df = extract_intent_features(test_n_df)
test_n_df = add_sarcasm(test_n_df)

## Add features

In [ ]:
EXT = ["has_elongation"]

MANUAL_LEX_FEAT_COLS = [
    "polite_phrase_cnt",
    "respect_title_cnt",
    "insult_cnt",
    "has_addressee",
]
MANUAL_EXT_LEX_FEAT_COLS = MANUAL_LEX_FEAT_COLS + EXT

AUTO_LEX_FEAT_COLS = [
    "polite_phrase_auto_cnt",
    "insult_auto_cnt",
    "neutral_auto_cnt",
]
AUTOL_EXT_LEX_FEAT_COLS = AUTO_LEX_FEAT_COLS + EXT

CLEAN_LEX_FEAT_COLS = [
    "polite_phrase_cnt_clean",
    "respect_title_cnt_clean",
    "insult_cnt_clean",
    "has_addressee_clean",
]
CLEAN_EXT_LEX_FEAT_COLS = CLEAN_LEX_FEAT_COLS + EXT

CLEAN_AUTO_LEX_FEAT_COLS = [
    "polite_phrase_auto_cnt_clean",
    "insult_auto_cnt_clean",
    "neutral_auto_cnt_clean",
]
CLEAN_AUTO_EXT_LEX_FEAT_COLS = CLEAN_AUTO_LEX_FEAT_COLS + EXT

PRAGMATIC_COLS = [
    'emoji_count',
    'repeated_punc',
    'exclamation_count',
    'arabic_question_count',
    'has_diacritics',
    'diacritic_count',
    'diacritic_density'
]

def build_feat_matrix(df_feat, cols):
    X = df_feat[cols].copy()
    X = X.fillna(0).astype(float)
    return csr_matrix(X.values)

Xtr_manual = build_feat_matrix(train_n_df, MANUAL_LEX_FEAT_COLS)
Xva_manual = build_feat_matrix(valid_n_df, MANUAL_LEX_FEAT_COLS)
Xte_manual = build_feat_matrix(test_n_df, MANUAL_LEX_FEAT_COLS)

Xtr_manual_ext = build_feat_matrix(train_n_df, MANUAL_EXT_LEX_FEAT_COLS)
Xva_manual_ext = build_feat_matrix(valid_n_df, MANUAL_EXT_LEX_FEAT_COLS)
Xte_manual_ext = build_feat_matrix(test_n_df, MANUAL_EXT_LEX_FEAT_COLS)

Xtr_auto = build_feat_matrix(train_n_df, AUTO_LEX_FEAT_COLS)
Xva_auto = build_feat_matrix(valid_n_df, AUTO_LEX_FEAT_COLS)
Xte_auto = build_feat_matrix(test_n_df, AUTO_LEX_FEAT_COLS)

Xtr_auto_ext = build_feat_matrix(train_n_df, AUTOL_EXT_LEX_FEAT_COLS)
Xva_auto_ext = build_feat_matrix(valid_n_df, AUTOL_EXT_LEX_FEAT_COLS)
Xte_auto_ext = build_feat_matrix(test_n_df, AUTOL_EXT_LEX_FEAT_COLS)

Xtr_manual_clean = build_feat_matrix(train_n_df, CLEAN_LEX_FEAT_COLS)
Xva_manual_clean = build_feat_matrix(valid_n_df, CLEAN_LEX_FEAT_COLS)
Xte_manual_clean = build_feat_matrix(test_n_df, CLEAN_LEX_FEAT_COLS)

Xtr_manual_clean_ext = build_feat_matrix(train_n_df, CLEAN_EXT_LEX_FEAT_COLS)
Xva_manual_clean_ext = build_feat_matrix(valid_n_df, CLEAN_EXT_LEX_FEAT_COLS)
Xte_manual_clean_ext = build_feat_matrix(test_n_df, CLEAN_EXT_LEX_FEAT_COLS)

Xtr_auto_clean = build_feat_matrix(train_n_df, CLEAN_AUTO_LEX_FEAT_COLS)
Xva_auto_clean = build_feat_matrix(valid_n_df, CLEAN_AUTO_LEX_FEAT_COLS)
Xte_auto_clean = build_feat_matrix(test_n_df, CLEAN_AUTO_LEX_FEAT_COLS)

Xtr_auto_clean_ext = build_feat_matrix(train_n_df, CLEAN_AUTO_EXT_LEX_FEAT_COLS)
Xva_auto_clean_ext = build_feat_matrix(valid_n_df, CLEAN_AUTO_EXT_LEX_FEAT_COLS)
Xte_auto_clean_ext = build_feat_matrix(test_n_df, CLEAN_AUTO_EXT_LEX_FEAT_COLS)

Xtr_prag = build_feat_matrix(train_n_df, PRAGMATIC_COLS)
Xva_prag = build_feat_matrix(valid_n_df, PRAGMATIC_COLS)
Xte_prag = build_feat_matrix(test_n_df, PRAGMATIC_COLS)

Xtr_ext = build_feat_matrix(train_n_df, EXT)
Xva_ext = build_feat_matrix(valid_n_df, EXT)
Xte_ext = build_feat_matrix(test_n_df, EXT)

In [ ]:
def encode_categorical_features(train_df, valid_df, test_df, column_name):
    tr = pd.get_dummies(train_df[column_name].astype(str), prefix=column_name)
    va = pd.get_dummies(valid_df[column_name].astype(str), prefix=column_name)
    te = pd.get_dummies(test_df[column_name].astype(str), prefix=column_name)

    tr, va = tr.align(va, join="outer", axis=1, fill_value=0)
    tr, te = tr.align(te, join="outer", axis=1, fill_value=0)

    va = va.reindex(columns=tr.columns, fill_value=0)
    te = te.reindex(columns=tr.columns, fill_value=0)

    train_df = pd.concat([train_df, tr], axis=1)
    valid_df = pd.concat([valid_df, va], axis=1)
    test_df  = pd.concat([test_df, te], axis=1)

    return train_df, valid_df, test_df, list(tr.columns)

train_n_df, valid_n_df, test_n_df, DIAL_COLS = encode_categorical_features(train_n_df, valid_n_df, test_n_df, 'dial_label')
train_n_df, valid_n_df, test_n_df, INTENT_COLS = encode_categorical_features(train_n_df, valid_n_df, test_n_df, 'intent_label')

DIALECT_FEAT_COLS = DIAL_COLS + ['dial_conf']
INTENT_FEAT_COLS = INTENT_COLS + ['intent_conf']
SAR_FEAT_COLS = ["sarcasm_score"]
Xtr_sarc = build_feat_matrix(train_n_df, SAR_FEAT_COLS)
Xva_sarc = build_feat_matrix(valid_n_df, SAR_FEAT_COLS)
Xte_sarc = build_feat_matrix(test_n_df, SAR_FEAT_COLS)

Xtr_dial = build_feat_matrix(train_n_df, DIALECT_FEAT_COLS)
Xva_dial = build_feat_matrix(valid_n_df, DIALECT_FEAT_COLS)
Xte_dial = build_feat_matrix(test_n_df, DIALECT_FEAT_COLS)

Xtr_intent = build_feat_matrix(train_n_df, INTENT_FEAT_COLS)
Xva_intent = build_feat_matrix(valid_n_df, INTENT_FEAT_COLS)
Xte_intent = build_feat_matrix(test_n_df, INTENT_FEAT_COLS)

In [ ]:
Xtr_sent_sp = csr_matrix(Xtr_sent)
Xva_sent_sp = csr_matrix(Xva_sent)

results_list = []

def fit_eval(Xtr, Xva, name, category="Experiment"):
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xva)
    score = f1_score(yva, pred, average="macro")
    acc = accuracy_score(yva, pred)
    results_list.append({
        "Category": category,
        "Experiment": name,
        "macroF1": score,
        "Acc": acc,
        "Dim": Xtr.shape[1]
    })
    print(f"[{category}] {name}: {score:.4f}")


# SECTION A: MAIN EXPERIMENTS
fit_eval(Xtr_sent_sp, Xva_sent_sp, "Baseline (MARBERT)", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_manual]),
         hstack([Xva_sent_sp, Xva_manual]), "Baseline + Manual", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_auto]),
         hstack([Xva_sent_sp, Xva_auto]), "Baseline + Auto", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_src]),
         hstack([Xva_sent_sp, Xva_src]), "Baseline + Source Data", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual, Xva_src]), "Baseline + Source Data + Manual", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_src]),
         hstack([Xva_sent_sp, Xva_auto, Xva_src]), "Baseline + Source Data + Auto", "Main")
print()

# SECTION B: ABLATION STUDY
# Trying Dialect
fit_eval(hstack([Xtr_sent_sp, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_dial]), "Ablation: Dialect (IbrahimAmin)", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_dial]), "Ablation: Manual + Dialect", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_auto, Xva_dial]), "Ablation: Auto + Dialect", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_dial]), "Ablation: Manual + Auto + Dialect", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Intent
fit_eval(hstack([Xtr_sent_sp, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_intent]), "Ablation: Intent (Bank77) ", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_intent]), "Ablation: Manual + Intent", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_auto, Xva_intent]), "Ablation: Auto + Intent", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_intent]), "Ablation: Manual + Auto + Intent", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Sarcasm
fit_eval(hstack([Xtr_sent_sp, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_sarc]), "Ablation: Sarcasm (hardiksr)", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc]), "Ablation: Manual + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_auto, Xva_sarc]), "Ablation: Auto + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_sarc]), "Ablation: Manual + Auto + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Elongations
fit_eval(hstack([Xtr_sent_sp, Xtr_ext]),
         hstack([Xva_sent_sp, Xva_ext]), "Ablation: Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext]),
         hstack([Xva_sent_sp, Xva_manual_ext]), "Ablation: Manual + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_ext]),
         hstack([Xva_sent_sp, Xva_auto_ext]), "Ablation: Auto + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_src]), "Ablation: Manual + Elongation + Source Data", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_ext, Xtr_src]),
         hstack([Xva_sent_sp, Xva_auto_ext, Xva_src]), "Ablation: Auto + Elongation + Source Data", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext]), "Ablation: Manual + Auto + Elongation", "Ablation: Preprocessing")
print()

# Trying cleaned up text
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean]),
         hstack([Xva_sent_sp, Xva_manual_clean]), "Ablation: Manual Clean", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean]),
         hstack([Xva_sent_sp, Xva_auto_clean]), "Ablation: Auto Clean", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext]), "Ablation: Manual Clean + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean_ext]),
         hstack([Xva_sent_sp, Xva_auto_clean_ext]), "Ablation: Auto Clean + Elongation", "Ablation: Preprocessing")
print()

# Trying other pragmatic features
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual_clean, Xva_prag]), "Ablation: Manual Clean + Pragmatic", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_auto_clean, Xva_prag]), "Ablation: Auto Clean + Pragmatic", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag]), "Ablation: Manual Clean + Elongation + Pragmatic + Elongation", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean_ext, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_auto_clean_ext, Xva_prag]), "Ablation: Auto Clean + Elongation + Pragmatic + Elongation", "Ablation: Pragmatics")

# Trying large
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc]), "Ablation: Manual Clean + Elongation + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc, Xva_dial]), "Ablation: Manual Clean + Elongation + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc, Xva_dial, Xva_intent]), "Ablation: Manual Clean + Elongation + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_sarc]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_sarc, Xva_dial]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_dial, Xva_intent]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_dial, Xtr_intent, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_dial, Xva_intent, Xva_src]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect + Intent + Source Data", "Ablation: All")
print()

fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc]), "Ablation: Manual + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc, Xva_dial]), "Ablation: Manual + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc, Xva_dial, Xva_intent]), "Ablation: Manual  Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag]), "Ablation: Manual + Pragmatic + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial, Xva_intent]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial, Xtr_intent, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial, Xva_intent, Xva_src]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect + Intent + Source Data", "Ablation: All")
print()

full_df = pd.DataFrame(results_list)

In [ ]:
import pandas as pd

full_df = pd.DataFrame(results_list)
full_df.to_csv('feature_fusion_experiments_MARBERT.csv')

baseline_val = full_df.loc[full_df['Experiment'] == 'Baseline (MARBERT)', 'macroF1'].values[0]
categories = [
    "Main",
    "Ablation: Preprocessing",
    "Ablation: Pragmatics",
    "Ablation: Intent, Dialect & Sarcasm",
    "Ablation: All"
]

for i, cat in enumerate(categories):
    if cat not in full_df['Category'].values:
        continue

    print(f"\nTABLE {i+1}: {cat.upper()} EXPERIMENTS (Sorted by Performance)")
    sub_df = full_df[full_df['Category'] == cat].copy()

    # Calculate improvement based on the global baseline
    sub_df['vs Baseline (MARBERT) (%)'] = ((sub_df['macroF1'] - baseline_val) / baseline_val) * 100
    styled_df = sub_df.sort_values(by='macroF1', ascending=False).style\
        .background_gradient(subset=['macroF1'], cmap='YlGn')\
        .format({'macroF1': '{:.3f}', 'Vs Ablation: Manual Clean + Elongation (%)': '{:+.2f}%'})\
        .hide(axis='index')

    display(styled_df)

# Fine-tune MARBERT with OPTUNA

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    default_data_collator
)

MODEL_NAME =  "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

MAX_LEN = 128
def tokenize(batch):
    return tokenizer(batch["Sentence"], truncation=True, max_length=MAX_LEN)

LABEL_ORDER = ["Impolite", "Neutral", "Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}
id2label = {i:l for l,i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
valid_df["label_id"] = valid_df["label"].map(label2id)
train_ds = Dataset.from_pandas(train_df[["Sentence", "label_id"]])
valid_ds = Dataset.from_pandas(valid_df[["Sentence", "label_id"]])


train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

cols_to_remove = ["Sentence", "__index_level_0__"] if "__index_level_0__" in train_ds.column_names else ["Sentence"]
train_ds = train_ds.remove_columns([c for c in cols_to_remove if c in train_ds.column_names])
valid_ds = valid_ds.remove_columns([c for c in cols_to_remove if c in valid_ds.column_names])

train_ds = train_ds.rename_column("label_id", "labels")
valid_ds = valid_ds.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed=44):
    """Sets the seed for reproducibility across python, numpy, and pytorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"> SEED SET TO: {seed}")

set_seed(42)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro
    }

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(list(id2label.keys())),
    y=train_df["label_id"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_ORDER),
        id2label=id2label,
        label2id=label2id
    )

args = TrainingArguments(
    output_dir="optuna_search",
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    logging_steps=50,
    fp16=True,
    report_to="none",
    disable_tqdm=False
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model_init=model_init,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 3),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.05, 0.15),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["cosine"]),
        "adam_epsilon": trial.suggest_float("adam_epsilon", 1e-8, 1e-6, log=True),
    }

In [ ]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=6
)
print("Best Trial Hyperparameters:", best_trial.hyperparameters)

In [ ]:
# UBC-NLP/MARBERTv2 0.847
best_params = {'learning_rate': 1.665325974338926e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.043076497700153246, 'num_train_epochs': 2, 'warmup_ratio': 0.09199190684733353, 'lr_scheduler_type': 'cosine', 'adam_epsilon': 2.421904688974844e-07}

In [ ]:
# best_params = best_trial.hyperparameters
print("Optimal Parameters Found:", best_params)

final_args = TrainingArguments(
    output_dir="marbert_final_gold",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["per_device_train_batch_size"],
    weight_decay=best_params["weight_decay"],
    num_train_epochs=best_params["num_train_epochs"],
    warmup_ratio=best_params["warmup_ratio"],
    lr_scheduler_type=best_params["lr_scheduler_type"],
    adam_epsilon=best_params["adam_epsilon"],

    # Essential for the final save
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    report_to="none"
)

final_trainer = WeightedTrainer(
    class_weights=class_weights,
    model_init=model_init,
    args=final_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
from scipy.special import softmax

pred = final_trainer.predict(valid_ds)
pred_labels = np.argmax(pred.predictions, axis=-1)
true_labels = pred.label_ids

logits = pred.predictions
p_marbert_valid = softmax(logits, axis=1)

report = classification_report(
    true_labels, pred_labels,
    target_names=[id2label[i] for i in range(len(LABEL_ORDER))],
    digits=3,
    zero_division=0
)
print(report)

In [ ]:
final_trainer.save_model("./best_marbert_model")

!zip -r fine_tuned_marbert.zip /kaggle/working/best_marbert_model

# Fine-tune MARBERT + Feature Fusion

In [ ]:
import torch
from tqdm import tqdm

def get_finetuned_embeddings(text_list, model, tokenizer):
    model.eval()
    all_embeddings = []

    # Process in batches for speed on Kaggle
    batch_size = 32
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch_text = text_list[i:i+batch_size]
        inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to("cuda")

        with torch.no_grad():
            # We take the [CLS] token (summary) from the last hidden layer
            outputs = model.bert(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

# Assuming 'trainer.model' is your best fine-tuned model
X_train_ft = get_finetuned_embeddings(train_df['Sentence'].tolist(), final_trainer.model, tokenizer)
X_valid_ft = get_finetuned_embeddings(valid_df['Sentence'].tolist(), final_trainer.model, tokenizer)
X_test_ft = get_finetuned_embeddings(test_df['Sentence'].tolist(), final_trainer.model, tokenizer)

In [ ]:
lr_hybrid = LogisticRegression(
    C=0.5, penalty="l1", solver="liblinear",
    max_iter=5000, class_weight="balanced", random_state=42
)

In [ ]:
Xtr_sent_sp = X_train_ft
Xva_sent_sp = X_valid_ft

results_list_ft = []

def fit_eval(Xtr, Xva, name, category="Experiment"):
    lr_hybrid.fit(Xtr, ytr)
    pred = lr_hybrid.predict(Xva)
    score = f1_score(yva, pred, average="macro")
    acc = accuracy_score(yva, pred)
    results_list_ft.append({
        "Category": category,
        "Experiment": name,
        "macroF1": score,
        "Acc": acc,
        "Dim": Xtr.shape[1]
    })
    print(f"[{category}] {name}: {score:.4f}")


# SECTION A: MAIN EXPERIMENTS
fit_eval(Xtr_sent_sp, Xva_sent_sp, "Baseline (MARBERT)", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_manual]),
         hstack([Xva_sent_sp, Xva_manual]), "Baseline + Manual", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_auto]),
         hstack([Xva_sent_sp, Xva_auto]), "Baseline + Auto", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_src]),
         hstack([Xva_sent_sp, Xva_src]), "Baseline + Source Data", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual, Xva_src]), "Baseline + Source Data + Manual", "Main")

fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_src]),
         hstack([Xva_sent_sp, Xva_auto, Xva_src]), "Baseline + Source Data + Auto", "Main")
print()

# SECTION B: ABLATION STUDY
# Trying Dialect
fit_eval(hstack([Xtr_sent_sp, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_dial]), "Ablation: Dialect (IbrahimAmin)", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_dial]), "Ablation: Manual + Dialect", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_auto, Xva_dial]), "Ablation: Auto + Dialect", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_dial]), "Ablation: Manual + Auto + Dialect", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Intent
fit_eval(hstack([Xtr_sent_sp, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_intent]), "Ablation: Intent (Bank77) ", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_intent]), "Ablation: Manual + Intent", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_auto, Xva_intent]), "Ablation: Auto + Intent", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_intent]), "Ablation: Manual + Auto + Intent", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Sarcasm
fit_eval(hstack([Xtr_sent_sp, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_sarc]), "Ablation: Sarcasm (hardiksr)", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc]), "Ablation: Manual + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_auto, Xva_sarc]), "Ablation: Auto + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext, Xva_sarc]), "Ablation: Manual + Auto + Sarcasm", "Ablation: Intent, Dialect & Sarcasm")
print()

# Trying Elongations
fit_eval(hstack([Xtr_sent_sp, Xtr_ext]),
         hstack([Xva_sent_sp, Xva_ext]), "Ablation: Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext]),
         hstack([Xva_sent_sp, Xva_manual_ext]), "Ablation: Manual + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_ext]),
         hstack([Xva_sent_sp, Xva_auto_ext]), "Ablation: Auto + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_src]), "Ablation: Manual + Elongation + Source Data", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_ext, Xtr_src]),
         hstack([Xva_sent_sp, Xva_auto_ext, Xva_src]), "Ablation: Auto + Elongation + Source Data", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_ext, Xtr_auto_ext]),
         hstack([Xva_sent_sp, Xva_manual_ext, Xva_auto_ext]), "Ablation: Manual + Auto + Elongation", "Ablation: Preprocessing")
print()

# Trying cleaned up text
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean]),
         hstack([Xva_sent_sp, Xva_manual_clean]), "Ablation: Manual Clean", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean]),
         hstack([Xva_sent_sp, Xva_auto_clean]), "Ablation: Auto Clean", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext]), "Ablation: Manual Clean + Elongation", "Ablation: Preprocessing")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean_ext]),
         hstack([Xva_sent_sp, Xva_auto_clean_ext]), "Ablation: Auto Clean + Elongation", "Ablation: Preprocessing")
print()

# Trying other pragmatic features
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual_clean, Xva_prag]), "Ablation: Manual Clean + Pragmatic", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_auto_clean, Xva_prag]), "Ablation: Auto Clean + Pragmatic", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag]), "Ablation: Manual Clean + Elongation + Pragmatic + Elongation", "Ablation: Pragmatics")
fit_eval(hstack([Xtr_sent_sp, Xtr_auto_clean_ext, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_auto_clean_ext, Xva_prag]), "Ablation: Auto Clean + Elongation + Pragmatic + Elongation", "Ablation: Pragmatics")

# Trying large
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc]), "Ablation: Manual Clean + Elongation + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc, Xva_dial]), "Ablation: Manual Clean + Elongation + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_sarc, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_sarc, Xva_dial, Xva_intent]), "Ablation: Manual Clean + Elongation + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_sarc]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_sarc, Xva_dial]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_dial, Xva_intent]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual_clean_ext, Xtr_prag, Xtr_dial, Xtr_intent, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual_clean_ext, Xva_prag, Xva_dial, Xva_intent, Xva_src]), "Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect + Intent + Source Data", "Ablation: All")
print()

fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc]), "Ablation: Manual + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc, Xva_dial]), "Ablation: Manual + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_sarc, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_sarc, Xva_dial, Xva_intent]), "Ablation: Manual  Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag]), "Ablation: Manual + Pragmatic + Sarcasm", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial, Xtr_intent]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial, Xva_intent]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect + Intent", "Ablation: All")
fit_eval(hstack([Xtr_sent_sp, Xtr_manual, Xtr_prag, Xtr_dial, Xtr_intent, Xtr_src]),
         hstack([Xva_sent_sp, Xva_manual, Xva_prag, Xva_dial, Xva_intent, Xva_src]), "Ablation: Manual + Pragmatic + Sarcasm + Dialect + Intent + Source Data", "Ablation: All")
print()

In [ ]:
import pandas as pd

full_df = pd.DataFrame(results_list_ft)
full_df_base = pd.DataFrame(results_list)
full_df.to_csv('feature_fusion_experiments_FTMARBERT.csv')

baseline_val = full_df_base.loc[full_df_base['Experiment'] == 'Ablation: Manual Clean + Elongation + Pragmatic + Sarcasm + Dialect', 'macroF1'].values[0]
categories = [
    "Main",
    "Ablation: Preprocessing",
    "Ablation: Pragmatics",
    "Ablation: Intent, Dialect & Sarcasm",
    "Ablation: All"
]

for i, cat in enumerate(categories):
    if cat not in full_df['Category'].values:
        continue

    print(f"\nTABLE {i+1}: {cat.upper()} EXPERIMENTS (Sorted by Performance)")
    sub_df = full_df[full_df['Category'] == cat].copy()

    # 2. Calculate improvement based on the global baseline
    sub_df['vs Best Frozen (Matryoshka) (%)'] = ((sub_df['macroF1'] - baseline_val) / baseline_val) * 100

    # 3. Render with styling
    styled_df = sub_df.sort_values(by='macroF1', ascending=False).style\
        .background_gradient(subset=['macroF1'], cmap='YlGn')\
        .format({'macroF1': '{:.3f}', 'Vs Ablation: Manual Clean + Elongation (%)': '{:+.2f}%'})\
        .hide(axis='index')

    display(styled_df)

# Ensembled model

## Finding Best Parameters

In [ ]:
import optuna
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.calibration import CalibratedClassifierCV

def tune_sgd_optuna(
    Xtr, ytr,
    Xva, yva,
    n_trials=40,
    seed=42,
    use_calibration=True,
    calib_method="sigmoid",
    calib_cv=3
):
    def objective(trial):
        # core SGD knobs that matter
        alpha = trial.suggest_float("alpha", 1e-6, 5e-2, log=True)
        penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
        l1_ratio = 0.15
        if penalty == "elasticnet":
            l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

        learning_rate = trial.suggest_categorical("learning_rate", ["optimal", "constant", "adaptive"])
        eta0 = 0.01
        if learning_rate in ["constant", "adaptive"]:
            eta0 = trial.suggest_float("eta0", 1e-4, 5e-1, log=True)

        max_iter = trial.suggest_int("max_iter", 400, 2500)
        tol = trial.suggest_float("tol", 1e-5, 1e-2, log=True)

        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

        base = SGDClassifier(
            loss="log_loss",
            alpha=alpha,
            penalty=penalty,
            l1_ratio=l1_ratio,
            learning_rate=learning_rate,
            eta0=eta0,
            max_iter=max_iter,
            tol=tol,
            early_stopping=False,
            random_state=seed
        )
        base.class_weight = class_weight

        # Fit + predict
        if use_calibration:
            # Calibration uses CV on TRAIN only (no leakage from VALID labels)
            clf = CalibratedClassifierCV(base, method=calib_method, cv=calib_cv)
            clf.fit(Xtr, ytr)
            pred = clf.predict(Xva)
        else:
            base.fit(Xtr, ytr)
            pred = base.predict(Xva)

        return f1_score(yva, pred, average="macro")

    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_params = study.best_params

    # Refit best model and return it
    alpha = best_params["alpha"]
    penalty = best_params["penalty"]
    l1_ratio = best_params.get("l1_ratio", 0.15)
    learning_rate = best_params["learning_rate"]
    eta0 = best_params.get("eta0", 0.01)
    max_iter = best_params["max_iter"]
    tol = best_params["tol"]
    class_weight = best_params["class_weight"]

    best_base = SGDClassifier(
        loss="log_loss",
        alpha=alpha,
        penalty=penalty,
        l1_ratio=l1_ratio,
        learning_rate=learning_rate,
        eta0=eta0,
        max_iter=max_iter,
        tol=tol,
        early_stopping=False,
        random_state=seed
    )
    best_base.class_weight = class_weight

    if use_calibration:
        best_clf = CalibratedClassifierCV(best_base, method=calib_method, cv=calib_cv)
        best_clf.fit(Xtr, ytr)
        best_pred = best_clf.predict(Xva)
    else:
        best_base.fit(Xtr, ytr)
        best_clf = best_base
        best_pred = best_base.predict(Xva)

    best_f1 = f1_score(yva, best_pred, average="macro")

    return study, best_params, best_f1, best_clf, best_pred

In [ ]:
from sklearn.preprocessing import LabelEncoder

Xtr_base = X_train_ft
Xva_base = X_valid_ft

le = LabelEncoder()
ytr_le = le.fit_transform(train_df["label"].values)
yva_le = le.transform(valid_df["label"].values)


Xtr = hstack([Xtr_base, Xtr_manual_clean_ext, Xtr_sarc])
Xva = hstack([Xva_base, Xva_manual_clean_ext, Xva_sarc])

study_sgd, best_sgd_params, best_sgd_f1, sgd_best_model, sgd_best_pred = tune_sgd_optuna(
    Xtr, ytr_le, Xva, yva_le,
    n_trials=30,
    use_calibration=True,
    calib_method="sigmoid",
    calib_cv=3
)
print("Best SGD Macro-F1:", round(best_sgd_f1, 4))
print("Best SGD params:", best_sgd_params)

In [ ]:
# sigmoid  0.8484
best_sgd_params = {'alpha': 0.0031560162444810784, 'penalty': 'elasticnet', 'l1_ratio': 0.06384409557707724, 'learning_rate': 'constant', 'eta0': 0.0004311429947100819, 'max_iter': 1660, 'tol': 0.00012843256723307116, 'class_weight': None}

In [ ]:
# from our grid search
best_lr_params = {'C': 0.5, 'penalty': 'l1', 'class_weight': 'balanced', 'fit_intercept': False}

## Funding Best Weights

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB

Xtr_base = X_train_ft
Xva_base = X_valid_ft

Xtr = hstack([Xtr_base, Xtr_manual_clean_ext, Xtr_sarc])
Xva = hstack([Xva_base, Xva_manual_clean_ext, Xva_sarc])

# Labels
le = LabelEncoder()
ytr_le = le.fit_transform(train_df["label"].values)
yva_le = le.transform(valid_df["label"].values)

neutral_id  = int(le.transform(["Neutral"])[0])
polite_id   = int(le.transform(["Polite"])[0])
impolite_id = int(le.transform(["Impolite"])[0])


lr = LogisticRegression(
    C=best_lr_params['C'],
    penalty=best_lr_params['penalty'],
    solver="liblinear",
    max_iter=5000,
    class_weight=best_lr_params['class_weight'],
    fit_intercept=best_lr_params['fit_intercept'],
    random_state=42
)
lr.fit(Xtr, ytr_le)
p_lr = lr.predict_proba(Xva)

sgd = SGDClassifier(
    loss="log_loss",
    alpha=best_sgd_params["alpha"],
    learning_rate=best_sgd_params["learning_rate"],
    eta0=best_sgd_params["eta0"],
    max_iter=best_sgd_params["max_iter"],
    penalty=best_sgd_params["penalty"],
    tol=best_sgd_params["tol"],
    l1_ratio=best_sgd_params["l1_ratio"],
    class_weight=best_sgd_params["class_weight"],
    random_state=42
)

sgd_cal = CalibratedClassifierCV(sgd, method="sigmoid", cv=3)
sgd_cal.fit(Xtr, ytr_le)
p_sgd = sgd_cal.predict_proba(Xva)

Xtr_nb = hstack([Xtr_manual_clean, Xtr_auto_clean, Xtr_dial])
Xva_nb = hstack([Xva_manual_clean, Xva_auto_clean, Xva_dial])

nb = ComplementNB(alpha=0.15)
nb.fit(Xtr_nb, ytr_le)
p_nb = nb.predict_proba(Xva_nb)

def apply_class_thresholds(p, t_neu=0.5, t_pol=0.20, t_imp=0.30):
    pred = np.full(p.shape[0], neutral_id, dtype=int)
    p_neu, p_pol, p_imp = p[:, neutral_id], p[:, polite_id], p[:, impolite_id]

    not_neu = p_neu < t_neu

    best_is_pol = p_pol >= p_imp
    best_cls = np.where(best_is_pol, polite_id, impolite_id)
    best_p   = np.where(best_is_pol, p_pol, p_imp)
    best_thr = np.where(best_is_pol, t_pol, t_imp)

    take_best = not_neu & (best_p >= best_thr)
    pred[take_best] = best_cls[take_best]
    return pred

# GRID SEARCH (weights + thresholds)
w_grid = np.arange(0.30, 0.91, 0.02)
t_neu_grid = np.arange(0.30, 0.51, 0.02)
t_pol_grid = np.arange(0.40, 0.71, 0.02)
t_imp_grid = np.arange(0.50, 0.71, 0.02)

best = (-1, None, None)
has_nb = p_nb is not None

for w_sgd  in w_grid:
    for w_lr in w_grid:
        if has_nb:
            w_nb = 1.0 - w_lr - w_sgd
            if w_nb < 0 or w_nb > 0.1:
                continue
        else:
            # renormalize LR/SGD only
            s = w_lr + w_sgd
            if s <= 0:
                continue
            w_lr2, w_sgd2 = w_lr/s, w_sgd/s

        # combine probs
        if has_nb:
            p_avg = w_lr*p_lr + w_sgd*p_sgd + w_nb*p_nb
        else:
            p_avg = w_lr2*p_lr + w_sgd2*p_sgd

        for t_neu in t_neu_grid:
            for t_pol in t_pol_grid:
                for t_imp in t_imp_grid:
                    pred = apply_class_thresholds(p_avg, t_neu=t_neu, t_pol=t_pol, t_imp=t_imp)
                    score = f1_score(yva_le, pred, average="macro")
                    if score > best[0]:
                        best = (score,
                                {"w_lr": float(w_lr), "w_sgd": float(w_sgd), "w_nb": float(1.0-w_lr-w_sgd) if has_nb else 0.0},
                                {"t_neu": float(t_neu), "t_pol": float(t_pol), "t_imp": float(t_imp)},
                                pred)

best_f1, best_w, best_t, best_pred = best
print("BEST Macro-F1:", round(float(best_f1), 2))
print("BEST weights:", best_w)
print("BEST thresholds:", best_t)
print("\nReport:")
print(classification_report(yva_le, best_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# 0.8595
BEST_W = {'w_lr': 0.4400000000000001, 'w_sgd': 0.46000000000000013, 'w_nb': 0.0999999999999997}
BEST_T = {'t_neu': 0.48000000000000015, 't_pol': 0.6600000000000003, 't_imp': 0.5}

# Multimodel Fusion Ensemble (MFE)

## ARM

In [ ]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, f1_score

LABEL_ORDER = ["Impolite", "Neutral", "Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}

yva = valid_df["label"].map(label2id).values

Xtr_base = X_train_ft
Xva_base = X_valid_ft

Xtr = hstack([Xtr_base, Xtr_manual_clean_ext, Xtr_sarc])
Xva = hstack([Xva_base, Xva_manual_clean_ext, Xva_sarc])

Xtr_nb = hstack([Xtr_manual_clean, Xtr_auto_clean, Xtr_dial])
Xva_nb = hstack([Xva_manual_clean, Xva_auto_clean, Xva_dial])

le = LabelEncoder()
le.fit(LABEL_ORDER)
ytr_le = le.transform(train_df["label"].values)

neutral_id  = int(le.transform(["Neutral"])[0])
polite_id   = int(le.transform(["Polite"])[0])
impolite_id = int(le.transform(["Impolite"])[0])

lr = LogisticRegression(
    C=best_lr_params['C'],
    penalty=best_lr_params['penalty'],
    solver="liblinear",
    max_iter=5000,
    class_weight=best_lr_params['class_weight'],
    fit_intercept=best_lr_params['fit_intercept'],
    random_state=42
)
lr.fit(Xtr, ytr_le)
p_lr = lr.predict_proba(Xva)

sgd = SGDClassifier(
    loss="log_loss",
    alpha=best_sgd_params["alpha"],
    learning_rate=best_sgd_params["learning_rate"],
    eta0=best_sgd_params["eta0"],
    max_iter=best_sgd_params["max_iter"],
    penalty=best_sgd_params["penalty"],
    tol=best_sgd_params["tol"],
    l1_ratio=best_sgd_params["l1_ratio"],
    class_weight=best_sgd_params["class_weight"],
    random_state=42
)

sgd_cal = CalibratedClassifierCV(sgd, method="sigmoid", cv=3)
sgd_cal.fit(Xtr, ytr_le)
p_sgd = sgd_cal.predict_proba(Xva)

nb = ComplementNB(alpha=0.15)
nb.fit(Xtr_nb, ytr_le)
p_nb = nb.predict_proba(Xva_nb)

# CLASSICAL ARM PROBS
w_lr, w_sgd, w_nb = BEST_W["w_lr"], BEST_W["w_sgd"], BEST_W["w_nb"]
p_classical_valid = (w_lr * p_lr) + (w_sgd * p_sgd) + (w_nb * p_nb)

# threshold postproc
def apply_class_thresholds(p, t_neu=0.56, t_pol=0.30, t_imp=0.42):
    pred = np.full(p.shape[0], neutral_id, dtype=int)
    p_neu, p_pol, p_imp = p[:, neutral_id], p[:, polite_id], p[:, impolite_id]
    not_neu = p_neu < t_neu
    best_is_pol = p_pol >= p_imp
    best_cls = np.where(best_is_pol, polite_id, impolite_id)
    best_p   = np.where(best_is_pol, p_pol, p_imp)
    best_thr = np.where(best_is_pol, t_pol, t_imp)
    take_best = not_neu & (best_p >= best_thr)
    pred[take_best] = best_cls[take_best]
    return pred

# Evaluate CLASSICAL ARM alone (no threshold)
pred_classical = p_classical_valid.argmax(axis=1)
print("\n[CLASSICAL ARM (prob argmax)] VALID")
print(classification_report(yva, pred_classical, target_names=LABEL_ORDER, digits=2))
print("Macro-F1:", f1_score(yva, pred_classical, average="macro"))

# classical with your thresholds
pred_classical_thr = apply_class_thresholds(
    p_classical_valid,
    t_neu=BEST_T["t_neu"], t_pol=BEST_T["t_pol"], t_imp=BEST_T["t_imp"]
)
print("\n[CLASSICAL ARM (thresholded)] VALID")
print(classification_report(yva, pred_classical_thr, target_names=LABEL_ORDER, digits=2))
print("Macro-F1:", f1_score(yva, pred_classical_thr, average="macro"))

## CAMeL

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from scipy.special import softmax
from sklearn.metrics import classification_report, f1_score

LABEL_ORDER = ["Impolite", "Neutral", "Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}
id2label = {i:l for l,i in label2id.items()}


MODEL_NAME ="CAMeL-Lab/bert-base-arabic-camelbert-da"
MAX_LEN = 128

# HF datasets
train_hf = Dataset.from_dict({
    "text": train_df["Sentence"].fillna("").astype(str).tolist(),
    "label": train_df["label"].map(label2id).tolist()
})
valid_hf = Dataset.from_dict({
    "text": valid_df["Sentence"].fillna("").astype(str).tolist(),
    "label": valid_df["label"].map(label2id).tolist()
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_hf.map(tok, batched=True, remove_columns=["text"])
valid_tok = valid_hf.map(tok, batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="camelbert_da_politeness",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    seed=42
)

trainer_camel = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer_camel.train()

In [ ]:
pred = trainer_camel.predict(valid_tok)
logits = pred.predictions
p_camel_valid = softmax(logits, axis=1)

print("CamelBERT id2label:", trainer_camel.model.config.id2label)
print("p_camel_valid shape:", p_camel_valid.shape)

yva = np.array(valid_hf["label"])
pred_camel = p_camel_valid.argmax(axis=1)

print("\n[CAMeLBERT-DA ALONE] VALID report")
print(classification_report(yva, pred_camel, target_names=LABEL_ORDER, digits=4))
print("Macro-F1:", round(float(f1_score(yva, pred_camel, average="macro")), 4))

## Matryoshka

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from scipy.special import softmax
from sklearn.metrics import classification_report, f1_score

LABEL_ORDER = ["Impolite", "Neutral", "Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}
id2label = {i:l for l,i in label2id.items()}

MODEL_NAME = "Omartificial-Intelligence-Space/Marbert-all-nli-triplet-Matryoshka"
MAX_LEN = 128

#  HF datasets
train_hf = Dataset.from_dict({
    "text": train_df["Sentence"].fillna("").astype(str).tolist(),
    "label": train_df["label"].map(label2id).tolist()
})
valid_hf = Dataset.from_dict({
    "text": valid_df["Sentence"].fillna("").astype(str).tolist(),
    "label": valid_df["label"].map(label2id).tolist()
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_hf.map(tok, batched=True, remove_columns=["text"])
valid_tok = valid_hf.map(tok, batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="matryoshka_politeness",
   eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    seed=42
)

trainer_xlmr = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer_xlmr.train()

In [ ]:
pred = trainer_xlmr.predict(valid_tok)
logits = pred.predictions
p_xlmr_valid = softmax(logits, axis=1)

yva = np.array(valid_hf["label"])
pred_xlmr = p_xlmr_valid.argmax(axis=1)

print("matryoshka id2label:", trainer_xlmr.model.config.id2label)
print("\n[matryoshka ALONE] VALID report")
print(classification_report(yva, pred_xlmr, target_names=LABEL_ORDER, digits=4))
print("Macro-F1:", round(float(f1_score(yva, pred_xlmr, average="macro")), 4))

print("\nSaved: p_xlmr_valid (N,3) for fusion")
print("Shape:", p_xlmr_valid.shape)

## Ensemble

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

LABEL_ORDER = ["Impolite","Neutral","Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}
yva = valid_df["label"].map(label2id).values

neutral_id, polite_id, impolite_id = 1, 2, 0

def apply_class_thresholds_fixedorder(p, t_neu=0.56, t_pol=0.30, t_imp=0.42):
    pred = np.full(p.shape[0], neutral_id, dtype=int)
    p_imp, p_neu, p_pol = p[:, impolite_id], p[:, neutral_id], p[:, polite_id]

    not_neu = p_neu < t_neu

    best_is_pol = p_pol >= p_imp
    best_cls = np.where(best_is_pol, polite_id, impolite_id)
    best_p   = np.where(best_is_pol, p_pol, p_imp)
    best_thr = np.where(best_is_pol, t_pol, t_imp)

    take_best = not_neu & (best_p >= best_thr)
    pred[take_best] = best_cls[take_best]
    return pred

# COLLECT ARMS
arms = [("cls", p_classical_valid), ("mar", p_marbert_valid)]
if "p_camel_valid" in globals() and globals()["p_camel_valid"] is not None:
    arms.append(("cam", globals()["p_camel_valid"]))
if "p_xlmr_valid" in globals() and globals()["p_xlmr_valid"] is not None:
    arms.append(("xlm", globals()["p_xlmr_valid"]))

for name, p in arms:
    assert p.shape == p_classical_valid.shape, f"{name} wrong shape {p.shape}"

arm_names = [a[0] for a in arms]
arm_probs  = [a[1] for a in arms]
K = len(arms)
print("Arms:", arm_names)

# WEIGHT GRID (small + sane)
w_mar_grid = [0.05, 0.10, 0.20, 0.30]
w_cam_grid = [0.05, 0.10, 0.15, 0.20]
w_xlm_grid = [0.05, 0.10, 0.15, 0.20]

# thresholds grid (modest)
t_neu_grid = np.arange(0.3, 0.41, 0.02)
t_pol_grid = np.arange(0.4, 0.71, 0.02)
t_imp_grid = np.arange(0.4, 0.71, 0.02)

# helper to fetch arm prob by name
P = {name: p for name, p in arms}

best = (-1.0, None, None, None)

for w_mar in w_mar_grid:
    for w_cam in (w_cam_grid if "cam" in P else [0.0]):
        for w_xlm in (w_xlm_grid if "xlm" in P else [0.0]):

            # remaining weight goes to classical (anchor)
            w_cls = 1.0 - (w_mar + w_cam + w_xlm)
            if w_cls < 0.3:   # force classical to remain dominant
                continue

            # fused probs
            p_fused = w_cls*P["cls"] + w_mar*P.get("mar", 0) + w_cam*P.get("cam", 0) + w_xlm*P.get("xlm", 0)

            for t_neu in t_neu_grid:
                for t_pol in t_pol_grid:
                    for t_imp in t_imp_grid:
                        pred = apply_class_thresholds_fixedorder(p_fused, t_neu=t_neu, t_pol=t_pol, t_imp=t_imp)
                        score = f1_score(yva, pred, average="macro")
                        if score > best[0]:
                            best = (
                                float(score),
                                {"w_cls": float(w_cls), "w_mar": float(w_mar), "w_cam": float(w_cam), "w_xlm": float(w_xlm)},
                                {"t_neu": float(t_neu), "t_pol": float(t_pol), "t_imp": float(t_imp)},
                                pred
                            )

best_f1, best_w, best_t, best_pred = best
print("\nBEST Macro-F1:", round(best_f1, 6))
print("BEST weights:", best_w)
print("BEST thresholds:", best_t)
print("\nReport:")
print(classification_report(yva, best_pred, target_names=LABEL_ORDER, digits=2))

In [ ]:
#  0.862
best_w = {'w_cls': 0.55, 'w_mar': 0.1, 'w_cam': 0.15, 'w_xlm': 0.2}
best_t = {'t_neu': 0.32, 't_pol': 0.4, 't_imp': 0.4}

# Submit-Valid

In [ ]:
import os, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report


TEAMNAME = "Anonymous_valid"
OUT_DIR  = "./"

print("Macro-F1:", round(best_f1, 4))
print("\nReport:")
print(classification_report(yva_le, best_pred, target_names=le.classes_, zero_division=0))

pred_labels = le.inverse_transform(best_pred)
sub = pd.DataFrame({"label": pred_labels})

csv_path = os.path.join(OUT_DIR, "subtask_A.csv")
zip_path = os.path.join(OUT_DIR, f"{TEAMNAME}.zip")

sub.to_csv(csv_path, index=False)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname="subtask_A.csv")

print("\nSaved:", csv_path)
print("Saved:", zip_path)

# Submit-Test

In [ ]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB

Xtr = hstack([X_train_ft, Xtr_manual_clean_ext, Xtr_sarc])
Xte = hstack([X_test_ft, Xte_manual_clean_ext, Xte_sarc])


Xtr_nb = hstack([Xtr_manual_clean_ext, Xtr_auto_clean, Xtr_dial])
Xte_nb = hstack([Xte_manual_clean_ext, Xte_auto_clean, Xte_dial])

le = LabelEncoder()
ytr = le.fit_transform(train_df["label"].values)


fixed = ["Impolite","Neutral","Polite"]
le_classes = list(le.classes_)
map_le_to_fixed = np.array([fixed.index(c) for c in le_classes], dtype=int)

lr = LogisticRegression(
    C=best_lr_params['C'],
    penalty=best_lr_params['penalty'],
    solver="liblinear",
    max_iter=5000,
    class_weight=best_lr_params['class_weight'],
    fit_intercept=best_lr_params['fit_intercept'],
    random_state=42
)
lr.fit(Xtr, ytr)
p_lr = lr.predict_proba(Xte)

sgd = SGDClassifier(
    loss="log_loss",
    alpha=best_sgd_params["alpha"],
    learning_rate=best_sgd_params["learning_rate"],
    eta0=best_sgd_params.get("eta0", 0.0),
    max_iter=best_sgd_params["max_iter"],
    penalty=best_sgd_params["penalty"],
    tol=best_sgd_params["tol"],
    l1_ratio=best_sgd_params.get("l1_ratio", 0.0),
    class_weight=best_sgd_params["class_weight"],
    random_state=42
)
sgd_cal = CalibratedClassifierCV(sgd, method="sigmoid", cv=3)
sgd_cal.fit(Xtr, ytr)
p_sgd = sgd_cal.predict_proba(Xte)

nb = ComplementNB(alpha=0.15)
nb.fit(Xtr_nb, ytr)
p_nb = nb.predict_proba(Xte_nb)

w_lr, w_sgd, w_nb = BEST_W["w_lr"], BEST_W["w_sgd"], BEST_W["w_nb"]
p_classical_test_le = (w_lr*p_lr + w_sgd*p_sgd + w_nb*p_nb)

p_classical_test = np.zeros_like(p_classical_test_le)
p_classical_test[:, map_le_to_fixed] = p_classical_test_le

print("p_classical_test shape:", p_classical_test.shape)

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import DataCollatorWithPadding
from scipy.special import softmax

LABEL_ORDER = ["Impolite","Neutral","Polite"]
label2id = {l:i for i,l in enumerate(LABEL_ORDER)}

def to_fixed_order_probs(p_any, model_id2label):
    """
    Reorder columns of probs to fixed LABEL_ORDER.
    model_id2label: dict {0:"Neutral",1:"Polite",2:"Impolite"} etc
    """
    idx_map = [None]*3
    for k, name in model_id2label.items():
        idx_map[label2id[name]] = int(k)
    if any(x is None for x in idx_map):
        raise ValueError(f"id2label mismatch: {model_id2label}")
    return p_any[:, idx_map]

def build_test_tok(test_df, tokenizer, max_len):
    test_hf = Dataset.from_dict({"text": test_df["Sentence"].fillna("").astype(str).tolist()})
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_len)
    test_tok = test_hf.map(tok, batched=True, remove_columns=["text"])
    return test_tok

In [ ]:
MAX_LEN_MAR = 128
mar_tokenizer = final_trainer.tokenizer

test_tok_mar = build_test_tok(test_df, mar_tokenizer, MAX_LEN_MAR)

pred_mar = final_trainer.predict(test_tok_mar)
p_any_mar = softmax(pred_mar.predictions, axis=1)

p_marbert_test = to_fixed_order_probs(p_any_mar, final_trainer.model.config.id2label)
print("p_marbert_test:", p_marbert_test.shape)

In [ ]:
MAX_LEN_CAM = 128
cam_tokenizer = trainer_camel.tokenizer

test_tok_cam = build_test_tok(test_df, cam_tokenizer, MAX_LEN_CAM)

pred_cam = trainer_camel.predict(test_tok_cam)
p_any_cam = softmax(pred_cam.predictions, axis=1)

p_camel_test = to_fixed_order_probs(p_any_cam, trainer_camel.model.config.id2label)
print("p_camel_test:", p_camel_test.shape)

In [ ]:
MAX_LEN_XLM = 128
xlm_tokenizer = trainer_xlmr.tokenizer

test_tok_xlm = build_test_tok(test_df, xlm_tokenizer, MAX_LEN_XLM)

pred_xlm = trainer_xlmr.predict(test_tok_xlm)
p_any_xlm = softmax(pred_xlm.predictions, axis=1)

p_xlmr_test = to_fixed_order_probs(p_any_xlm, trainer_xlmr.model.config.id2label)
print("p_xlmr_test:", p_xlmr_test.shape)

In [ ]:
import numpy as np
import pandas as pd

LABEL_ORDER = ["Impolite", "Neutral", "Polite"]
id2label = {0:"Impolite", 1:"Neutral", 2:"Polite"}

impolite_id, neutral_id, polite_id = 0, 1, 2

def apply_class_thresholds_fixedorder(p, t_neu=0.52, t_pol=0.50, t_imp=0.50):
    pred = np.full(p.shape[0], neutral_id, dtype=int)
    p_imp, p_neu, p_pol = p[:, impolite_id], p[:, neutral_id], p[:, polite_id]

    not_neu = p_neu < t_neu

    best_is_pol = p_pol >= p_imp
    best_cls = np.where(best_is_pol, polite_id, impolite_id)
    best_p   = np.where(best_is_pol, p_pol, p_imp)
    best_thr = np.where(best_is_pol, t_pol, t_imp)

    take_best = not_neu & (best_p >= best_thr)
    pred[take_best] = best_cls[take_best]
    return pred

# COLLECT TEST ARMS
Ptest = {"cls": p_classical_test}

if "p_marbert_test" in globals() and globals()["p_marbert_test"] is not None:
    Ptest["mar"] = globals()["p_marbert_test"]

if "p_camel_test" in globals() and globals()["p_camel_test"] is not None:
    Ptest["cam"] = globals()["p_camel_test"]

if "p_xlmr_test" in globals() and globals()["p_xlmr_test"] is not None:
    Ptest["xlm"] = globals()["p_xlmr_test"]

# sanity shape checks
n_test = Ptest["cls"].shape[0]
for k, p in Ptest.items():
    assert p.shape == (n_test, 3), f"{k} has wrong shape {p.shape}, expected {(n_test,3)}"

print("TEST arms available:", sorted(Ptest.keys()))
print("n_test:", n_test)

# FUSE WITH FROZEN WEIGHTS
w_cls = best_w.get("w_cls", 1.0)
w_mar = best_w.get("w_mar", 0.0)
w_cam = best_w.get("w_cam", 0.0)
w_xlm = best_w.get("w_xlm", 0.0)

used = []
w_used = []
for name, w in [("cls", w_cls), ("mar", w_mar), ("cam", w_cam), ("xlm", w_xlm)]:
    if w > 0 and name in Ptest:
        used.append(name)
        w_used.append(w)

w_used = np.array(w_used, dtype=float)
w_used = w_used / w_used.sum()

p_fused_test = np.zeros((n_test, 3), dtype=float)
for name, w in zip(used, w_used):
    p_fused_test += w * Ptest[name]

print("Using arms:", used)
print("Renormalized weights:", {n: float(w) for n, w in zip(used, w_used)})

# APPLY FROZEN THRESHOLDS
pred_test_ids = apply_class_thresholds_fixedorder(
    p_fused_test,
    t_neu=float(best_t["t_neu"]),
    t_pol=float(best_t["t_pol"]),
    t_imp=float(best_t["t_imp"])
)

pred_test_labels = np.array([id2label[i] for i in pred_test_ids])
print("Pred label counts:", dict(zip(*np.unique(pred_test_labels, return_counts=True))))

In [ ]:
import os, zipfile
import numpy as np
import pandas as pd

TEAMNAME = "Anonymous_test"
OUT_DIR  = "./"

sub = pd.DataFrame({"label": pred_test_labels})

csv_path = os.path.join(OUT_DIR, "subtask_A.csv")
zip_path = os.path.join(OUT_DIR, f"{TEAMNAME}.zip")

sub.to_csv(csv_path, index=False)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname="subtask_A.csv")

print("\nSaved:", csv_path)
print("Saved:", zip_path)

sub = pd.DataFrame({"label": pred_test_labels})
sub.to_csv("subtask_A.csv", index=False)
print("Saved: subtask_A.csv")
print(sub.head())